# Exponential Regression with Threshold Testers

This notebook implements an **Exponential Regression** pipeline to classify ordinal text categories. 

Exponential regression models the relationship $y = a e^{bx}$. By taking the natural logarithm of both sides, this becomes a linear relationship: $\ln(y) = \ln(a) + bx$. 

**The Pipeline:**
1. **Target Encoding:** Convert string labels from the original dataset into numeric ordinal values.
2. **Log-Transformation:** We apply `np.log1p` (log(y+1)) to our numeric targets to linearize the exponential relationship.
3. **Modeling:** Use TF-IDF and a Linear `Ridge` Regressor optimized via `GridSearchCV`.
4. **Inverse Transformation:** The model automatically applies `np.expm1` to exponentiate the continuous predictions back to the original scale.
5. **Dynamic Cutoff Testers:** We use a coordinate-descent algorithm to iteratively find the best threshold cutoffs to map the continuous exponential predictions back into discrete categories, regardless of how many classes exist in the original data.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import accuracy_score, classification_report

# Set random seed for reproducibility
SEED = 42

## 1. Load the Original Dataset & Encode Labels
Since this is the original dataset (`../data/train_noDuplicates.csv`), the labels might be text strings. We encode them into integers $0, 1, \dots, N$ to represent the categorical hierarchy.

In [2]:
# Load the dataset verbatim
df = pd.read_csv('/kaggle/input/datasets/juliaduboisas/train-csv/../data/train.csv')

# Drop any rows with missing text or labels
df = df.dropna(subset=['resp_text', 'clarity'])

X_raw = df['resp_text'].astype(str)
labels = df['clarity']

# Dynamically encode categories to integers (0, 1, 2, ...)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)

print(f"Loaded {len(X_raw)} samples.")
print(f"Found {num_classes} categories: {dict(enumerate(label_encoder.classes_))}")

# Split into Train, Validation (for tuning cutoffs), and Test sets
X_temp, X_test, y_temp, y_test = train_test_split(X_raw, y_encoded, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15, random_state=SEED)

print(f"Training set: {len(X_train)}")
print(f"Validation set: {len(X_val)}")
print(f"Test set: {len(X_test)}")

Loaded 20092 samples.
Found 3 categories: {0: 'c1', 1: 'c234', 2: 'c5'}
Training set: 14516
Validation set: 2562
Test set: 3014


## 2. Exponential Regression via Target Transformation & Grid Search
We use `TransformedTargetRegressor` to handle the exponential math under the hood. It takes the log of the targets during training and exponentiates the predictions during inference. We wrap this in a Cross-Validated Grid Search.

In [3]:
# Create the Ridge regressor
ridge = Ridge(random_state=SEED)

# Wrap Ridge in a TransformedTargetRegressor for Exponential Regression
# np.log1p handles log(y + 1) safely if there are 0s; np.expm1 reverses it
exp_regressor = TransformedTargetRegressor(
    regressor=ridge, 
    func=np.log1p, 
    inverse_func=np.expm1
)

# Build the pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('model', exp_regressor)
])

# Define hyperparameters for Grid Search
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'model__regressor__alpha': [0.1, 1.0, 10.0]
}

print("Running Grid Search with 5-Fold Cross Validation...")
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1,
    verbose=1
)

# Train the model
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print("\nBest Parameters found:")
print(grid_search.best_params_)

Running Grid Search with 5-Fold Cross Validation...
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Best Parameters found:
{'model__regressor__alpha': 10.0, 'tfidf__ngram_range': (1, 1)}


## 3. Dynamic Threshold Testers (Coordinate Descent)
The model outputs continuous exponential predictions. We need $N-1$ cutoffs to separate $N$ classes. 

This algorithm starts with mathematical midpoints (0.5, 1.5, 2.5...) and iteratively adjusts each cutoff slightly to see what maximizes **Validation Accuracy**.

In [4]:
# Generate continuous validation predictions
val_preds = best_model.predict(X_val)

def assign_classes_dynamic(preds, cutoffs):
    """Maps continuous predictions to discrete classes based on an array of cutoffs."""
    classes = np.zeros_like(preds, dtype=int)
    for i, cutoff in enumerate(cutoffs):
        classes[preds > cutoff] = i + 1
    return classes

# Initialize cutoffs at the mathematical midpoints
best_cutoffs = [i + 0.5 for i in range(num_classes - 1)]
best_acc = 0

print(f"Initial Baseline Cutoffs: {best_cutoffs}")

# Coordinate Descent Optimization: Tune each cutoff iteratively
epochs = 3 
for epoch in range(epochs):
    for i in range(len(best_cutoffs)):
        # Search space: +/- 0.5 around the current best cutoff in increments of 0.05
        search_space = np.linspace(best_cutoffs[i] - 0.5, best_cutoffs[i] + 0.5, 21)
        
        for candidate in search_space:
            temp_cutoffs = best_cutoffs.copy()
            temp_cutoffs[i] = candidate
            
            # Ensure cutoffs remain strictly ascending (no overlapping boundaries)
            if temp_cutoffs != sorted(temp_cutoffs):
                continue
                
            # Discretize and score
            preds_mapped = assign_classes_dynamic(val_preds, temp_cutoffs)
            acc = accuracy_score(y_val, preds_mapped)
            
            # Update if we find a better cutoff
            if acc > best_acc:
                best_acc = acc
                best_cutoffs = temp_cutoffs.copy()

# Round optimized cutoffs for cleaner output
best_cutoffs = [round(c, 3) for c in best_cutoffs]

print(f"Optimized Cutoffs found: {best_cutoffs}")
print(f"Validation Accuracy with optimized cutoffs: {best_acc:.4f}")

Initial Baseline Cutoffs: [0.5, 1.5]
Optimized Cutoffs found: [np.float64(0.85), np.float64(0.95)]
Validation Accuracy with optimized cutoffs: 0.4516


## 4. Final Evaluation on Unseen Test Data
We apply our dynamically optimized cutoffs to the test set and map the integer predictions back to their original string categories for evaluation.

In [5]:
# 1. Generate continuous predictions on the test set
test_preds = best_model.predict(X_test)

# 2. Map continuous values to integer classes using the optimized cutoffs
test_classes_int = assign_classes_dynamic(test_preds, best_cutoffs)

# Clip the predictions just in case extreme values exceeded the class bounds
test_classes_int = np.clip(test_classes_int, 0, num_classes - 1)

# 3. Revert integer classes back to the original string labels
final_predictions = label_encoder.inverse_transform(test_classes_int)
original_y_test = label_encoder.inverse_transform(y_test)

# 4. Evaluate and print metrics
print("Final Test Accuracy:", accuracy_score(original_y_test, final_predictions))
print("\nClassification Report:\n")
print(classification_report(original_y_test, final_predictions))

Final Test Accuracy: 0.44790975447909753

Classification Report:

              precision    recall  f1-score   support

          c1       0.42      0.66      0.52       922
        c234       0.37      0.19      0.25      1044
          c5       0.53      0.52      0.52      1048

    accuracy                           0.45      3014
   macro avg       0.44      0.46      0.43      3014
weighted avg       0.44      0.45      0.43      3014

